In [8]:
# Cell 1: setup. autoreload re-reads models/*.py whenever you save them,
# so you never need to restart the kernel after editing.
%load_ext autoreload
%autoreload 2

import pandas as pd
from models import common

# REAL data: data/mtg_data_clean.csv, one row per deck
decks = common.load_decks()
# FAKE data with a known answer (2.9 mana value, 17 lands), to test the code:
# decks = common.fake_decks()

# hold out a random 25% of drafts (seed 0) as the test set; train on the rest
train, test = common.split_by_draft(decks)
print(len(decks), "decks:", len(train), "train,", len(test), "test")
decks.head()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
106900 decks: 80264 train, 26636 test


,deck_id,draft_id,build_index,main_colors,wins,games,losses,num_cards,total_mana_value,num_lands,num_creatures,num_spells,avg_mana_value
0,deck_1,488a8410974649eb82add4b1a6f54a75,0,BR,3,6,3,40,69,16,16,24,2.875000
1,deck_2,41dafe993991494e9bbf2894af398d25,0,WR,6,9,3,40,55,16,15,24,2.291667
2,deck_3,f570c6a0e89a44ff83f9301b1fa8ef1b,0,WB,2,5,3,40,66,17,13,23,2.869565
3,deck_4,7c5e0736075241d7a0170c535f9a0cdf,0,WR,3,6,3,40,48,16,13,24,2.000000
4,deck_5,7d6e7ac9a9eb40808c54bd251b72837c,1,RG,5,8,3,40,66,17,18,23,2.869565


In [9]:
# Cell 2: baseline = predict the average win rate for every deck
from models import baseline
base = baseline.fit(train)
print("baseline win rate:", base)
print("baseline error:", common.error_per_100(test, baseline.predict(base, test)))

baseline win rate: 0.5575263613930368
baseline error: 17.491505726431143


In [ ]:
# Cell 3: main model, logistic regression in statsmodels
from models import statsmodels_logit
sml = statsmodels_logit.fit(train)
best = statsmodels_logit.best_values(sml, train)
for col, b in best.items():
    print(col, "| best:", b["best"], "| shape:", b["shape"])
print("error:", common.error_per_100(test, statsmodels_logit.predict(sml, test)))

In [11]:
# Cell 4: SGD at three learning rates (the slides' lesson: too small is slow)
from models import sgd_logit
for alpha in [0.01, 0.1, 0.5]:
    f = sgd_logit.fit(train, alpha=alpha)
    print(alpha, common.error_per_100(test, sgd_logit.predict(f, test)))

0.01 17.42205742090771
0.1 17.448529845006004
0.5 17.550325292718316


In [12]:
# Cell 5: the final results, all in one place
from models import report
results = report.final_results(train, test)

FINAL RESULTS
Training decks: 80264   Test decks: 26636

1. The model
   logit(P(win)) = -10.156 -0.038*mana -0.010*mana^2 +0.006*creatures +1.298*lands -0.040*lands^2 + colour pair effect

2. Sweet spots (a real peak needs: curve bends down, p < 0.05, inside the data)
   mana value: peak at -1.88, p = 0.719, real peak: False
   lands:      peak at 16.26, p = 0.000, real peak: True

3. Plain-language effect
   one more creature is about +0.2 wins per 100 games (near a 50% deck)

4. How wrong is each method on the test decks? (wins per 100 games, smaller is better)
   baseline (average win rate)               17.49
   statsmodels logistic regression           17.42
   SGD logistic regression                   17.55

5. What the advice is worth (test decks, wins per 100 games)
   no value score: there is no real sweet spot to build toward

6. The rule
   Mana value: no sweet spot found; the model gives no mana-value advice.
   Play 16 lands.
